# Заняття 5 · Демо 3 · Текстовий пошук по аеровідео: від архіву до вузького каналу

**LIVE DEMO 3 (слайд 44)** · слайди 22–23, 38, 41–43, 46, 48–51

Оператору треба знайти у відеоархіві «вантажівку біля будівлі», не переглядаючи години записів. Будуємо повний конвеєр
і чесно міряємо, що працює: вебова модель (SigLIP 2) проти доменних (RemoteCLIP, SkyCLIP, слайд 38) на тих самих кадрах.

| Розділ | Що показуємо | Слайди |
|---|---|---|
| 1. Дані | Ключові кадри, дедуплікація, тайли: скільки пікселів бачить модель | 22, 23, 50 |
| 2. Ground truth | Події замість кадрів і частка кадрів з ціллю | 46 |
| 3. Індекс | Ембеддинги тайлів, FAISS проти numpy | 43 |
| 4. Текстовий пошук | Запит → ранжовані кадри з таймкодами (панель) | 41 |
| 5. Кадр + текст | Пошук за зразком, modality gap і злиття оцінок з вагою λ (панель) | 34, 42 |
| 6. Метрики | EventRecall@k, P@k: промпт plain / aerial, кадр / тайли | 23, 38, 46 |
| 7. Три зони | Пороги по сирому cos і навантаження на оператора (панель) | 48, 49 |
| 8. Вузький канал | PQ-коди замість векторів: скільки пошуку лишається в 1.5–6 кбіт/с | 50, 51 |

**Підготовка (один раз, з мережею):**
```bash
# з кореня репозиторію
python tools/fetch_clip_models.py     # ваги моделей -> artifacts/models/clip/ (+ manifest.json з SHA-256)
python tools/fetch_visdrone.py --splits train val test-dev   # VisDrone-VID -> artifacts/visdrone/ (~11 ГБ)
```
Перший прогін рахує ембеддинги кількох десятків тисяч тайлів (хвилини на GPU), далі все береться з кешу `outputs/demo3/`.

In [ ]:
import os, time
from pathlib import Path

SMOKE = os.environ.get("DEMO3_SMOKE") == "1"    # синтетичні кадри й випадкові ваги: перевірка, що все запускається
if SMOKE:
    os.environ.setdefault("PANELS_AUTORUN", "0")   # nbconvert: панелі без автозапуску (див. panels.AUTORUN)

# ---------------- Конфігурація ----------------
DATASET = "visdrone"            # "visdrone" — авто-GT з розмітки | "own" — своє відео + gt/own_queries.csv
VISDRONE_SPLITS = ["train", "val", "test-dev"]
OWN_GT_CSV = Path("gt/own_queries.csv")          # шаблон: gt/own_queries_template.csv
MODELS = ["siglip2", "remoteclip", "skyclip"]
SIZE = "base"                   # "base": SigLIP 2 B/16, RemoteCLIP B/32, SkyCLIP B/32 | "large": L/16, L/14, L/14
SAMPLE_FPS = 3.0                # VisDrone — хвилини відео, беремо щільніше; для години власного польоту — 1.0
DEDUP_THR = 2.0                 # 0 — без дедуплікації (статична камера: рухома ціль мініатюру майже не змінює)
TILE_GRID = (2, 3)              # None — лише увесь кадр
TILE_OVERLAP = 0.15
MIN_PX = 24                     # мінімальна коротка сторона цілі (px вихідного кадру) для GT
DEVICE = None                   # None — cuda / mps / cpu автоматично

import panels as P

VISDRONE_DIR, CACHE_DIR = P.VISDRONE_DIR, Path("outputs/demo3")
if SMOKE:
    VISDRONE_DIR, CACHE_DIR = Path("outputs/smoke3/visdrone"), Path("outputs/smoke3/cache")
    DATASET, SAMPLE_FPS, DEDUP_THR, MIN_PX = "visdrone", 5.0, 0.0, 20
hf = "smoke" if SMOKE else P.setup_hf(clip=MODELS, size=SIZE)   # офлайн, якщо всі ваги вже в маніфесті

import numpy as np, pandas as pd
from PIL import Image
pd.set_option("display.precision", 3)
print(f"dataset: {DATASET} | models: {MODELS} {SIZE} | HF: {hf}")

## 1. Дані: ключові кадри й тайли

Проріджуємо відео до `SAMPLE_FPS` і відкидаємо майже однакові кадри (слайд 50 «Вузький канал»: «відбір ключових кадрів і дедуплікація»).
Дедуплікація не залежить від моделі, тож усі моделі індексують **той самий** набір кадрів.

Модель бачить 224×224. Кадр 1920 px, стиснутий до 224, — це ×8.6: ціль 40 px стає ~5 px (слайд 22 «Розрахунок, який треба зробити до вибору архітектури»). Тому індексуємо
**увесь кадр (тайл 0) + сітку тайлів з перекриттям** (тайлінг SAHI, слайд 23 «Архітектурні стратегії для високої роздільності»), а оцінка кадру — максимум по його тайлах.

In [ ]:
if SMOKE:
    P.make_synthetic_visdrone(VISDRONE_DIR)
elif DATASET == "visdrone":
    P.ensure_visdrone(VISDRONE_SPLITS, VISDRONE_DIR)
sources = P.discover_visdrone(VISDRONE_DIR) if DATASET == "visdrone" else P.discover_generic(P.OWN_VIDEO_DIR)
assert sources, "Джерел не знайдено"
total_s = sum(P.source_seconds(s) for s in sources)

kf = P.extract_keyframes(sources, CACHE_DIR / "keyframes", sample_fps=SAMPLE_FPS, dedup_thr=DEDUP_THR)
tiles = P.make_tiles(kf, TILE_GRID, TILE_OVERLAP)
tpf = int(tiles.groupby("kf_id").size().max())
st, w_med = kf.attrs["stats"], int(kf.width.median())
tile_w = int((tiles[tiles.tile_idx > 0].x1 - tiles[tiles.tile_idx > 0].x0).median()) if TILE_GRID else w_med
print(f"Джерел: {len(sources)} · відео {total_s / 60:.1f} хв · ключових кадрів {st['n_kept']} з {st['n_seen']} переглянутих"
      + (" (з кешу)" if st["cached"] else ""))
print(f"Тайлів: {len(tiles)} ({tpf} на кадр) · кадр ~{w_med}px → 224: ×{w_med / 224:.1f} · тайл ~{tile_w}px → 224: ×{tile_w / 224:.1f}")
P.show_tiles(kf.iloc[len(kf) // 2], tiles, "Тайл 0 = увесь кадр; 1..N — сітка з перекриттям")

**Підсумок розділу.** Пікселі на цілі рахуємо після всіх ресайзів: тайли зберігають нативний GSD ціною в `tpf` разів більше
ембеддингів (слайд 23).

## 2. Ground truth

**VisDrone:** кадр релевантний запиту `truck`, якщо на ньому є вантажівка з короткою стороною ≥ `MIN_PX` без сильної оклюзії.
Сусідні релевантні кадри одного кліпу зливаються в **подію** — одну ціль, яку оператор має побачити хоча б раз.
**Власне відео:** `artifacts/aerial_video/*.mp4` + `gt/own_queries.csv`, рядок на кожен інтервал, де ціль видно.

In [ ]:
gt = P.load_visdrone_gt(kf, sources, min_px=MIN_PX) if DATASET == "visdrone" else P.load_csv_gt(kf, OWN_GT_CSV)
gt = {q: g for q, g in gt.items() if g["events"]}
assert gt, "Немає жодного запиту з подіями — перевірте GT"
display(P.gt_summary(gt, len(kf)))

**Підсумок розділу.** `prevalence` — частка кадрів з ціллю. Якщо ціль у 1 % кадрів, випадкове ранжування дає P@10 ≈ 0.01:
метрики порівнюємо з цим числом, а не з нулем (слайд 46).

## 3. Ембеддинги й індекс FAISS

Усі три моделі вантажаться через open_clip, тож інтерфейс однаковий. Кадр подається **letterbox**, а не center-crop:
інакше з кадру 16:9 зникає 44 % ширини (слайд 22). SigLIP 2 B/**16** має 196 патчів, а B/**32** — 49: це порівняння
«що взяти з полиці», а не методів навчання за рівних умов.

In [ ]:
enc, embs, indices, rows = {}, {}, {}, []
for name in MODELS:
    t0 = time.time()
    enc[name] = P.ClipEncoder(name, SIZE, device=DEVICE, random_weights=SMOKE)
    t_load = time.time() - t0
    embs[name] = P.embed_tiles(enc[name], kf, tiles, CACHE_DIR / "emb")     # кеш + продовження обірваного прогону
    indices[name] = P.build_flat_index(embs[name])
    rows.append(dict(model=name, arch=enc[name].arch, params_M=enc[name].n_params / 1e6, dim=enc[name].dim,
                     vectors=len(embs[name]), img_per_s=enc[name].img_per_s, load_s=t_load, cached=enc[name].cached))
pd.DataFrame(rows)

### FAISS чи numpy? (слайд 43 «FAISS: векторна індексація»)

Точний пошук (`IndexFlatIP`) — та сама операція, що й `E @ q`: множення матриці на вектор і відбір топу. Порівнюємо
повне сортування, частковий відбір `argpartition` і FAISS — на нашому архіві й на синтетичних 200 тис. векторах.

In [ ]:
K = 50
E0 = np.ascontiguousarray(embs[MODELS[0]], dtype="float32")
q = enc[MODELS[0]].text_query("a truck")


def bench(fn, reps=30):
    fn()
    t0 = time.perf_counter()
    for _ in range(reps):
        fn()
    return (time.perf_counter() - t0) / reps * 1e3


def methods(E, index, Q):
    return {"numpy argsort": lambda: np.argsort(-(Q @ E.T), axis=1)[:, :K],
            "numpy argpartition": lambda: np.argpartition(-(Q @ E.T), K, axis=1)[:, :K],
            "FAISS IndexFlatIP": lambda: index.search(Q, K)}


rows = []
E_big = P.l2n(np.random.default_rng(1).standard_normal((20_000 if SMOKE else 200_000, E0.shape[1]), dtype=np.float32))
for label, E, index in [(f"{len(E0):,} (архів)", E0, indices[MODELS[0]]), (f"{len(E_big):,} (синтетика)", E_big, P.build_flat_index(E_big))]:
    for qname, Q in [("1 запит", q[None]), ("64 запити", E[np.random.default_rng(0).choice(len(E), 64)])]:
        for m, fn in methods(E, index, np.ascontiguousarray(Q)).items():
            rows.append(dict(vectors=label, queries=qname, method=m, ms_per_query=bench(fn) / len(Q)))
del E_big
pd.DataFrame(rows).pivot_table(index=["vectors", "queries"], columns="method", values="ms_per_query", sort=False).round(3)

**Підсумок розділу.** На точному пошуку FAISS не виграє: вирішує алгоритм (`argpartition` за O(N) замість сортування,
пакет запитів замість одиночних). FAISS потрібен, коли точний перебір уже не вміщається в бюджет: **наближені індекси**
(IVF, HNSW), **PQ-стиснення** (розділ 8), GPU, збереження індексу на диск. Пам'ять: 512 × FP16 = 1 КБ на вектор (слайд 43); у SigLIP 2 B/16 вектор 768-вимірний — 1.5 КБ.
Таймінги чесні лише на вільній машині.

## 4. Текстовий пошук (слайд 41 «Text-to-Image»)

Оберіть запит із GT (з'являться позначки ✓ / ✗ і P@k) або впишіть свій. Жовта рамка — тайл, що дав максимальний cos.
Пишіть позитивні описи з ракурсом; заперечення, рахунок і «ліворуч від» не працюють (Демо 2, батарея відмов).

In [ ]:
def ask(text, k=8, use_tiles=True, aerial=True, models=None, gt_query=None):
    rel = gt[gt_query]["relevant"] if gt_query else None
    for name in models or MODELS:
        qv = enc[name].text_query(text, P.AERIAL_TEMPLATES if aerial else P.PLAIN_TEMPLATES)
        res = P.top_frames(P.all_tile_scores(indices[name], qv), tiles, kf, k=k, use_tiles=use_tiles)
        hit = f" | P@{k} = {np.mean([r in rel for r in res.kf_id]):.2f}" if rel is not None else ""
        P.show_results(res, f"{name}: «{text}» {'(тайли)' if use_tiles else '(увесь кадр)'}{hit}", relevant=rel)


demo_q = max(gt, key=lambda q: len(gt[q]["events"]))
P.search_box(ask, MODELS, gt_texts={q: g["text"] for q, g in gt.items()}, gt_query=demo_q)

**Підсумок розділу.** Текстовий запит звужує архів до кількох десятків кадрів без жодної розмітки. Але «красива» перша
видача ще нічого не доводить — числа будуть у розділі 6.

## 5. Кадр + текст → кадр (слайди 34, 42 «Image+Text-to-Image»)

Оператор знайшов одну ціль і хоче решту схожих — **пошук за зразком**. А якщо треба «таку саму сцену, але з машинами»?
Наївна сума векторів `кадр + текст` не працює через **modality gap**: кадри між собою схожі з cos ~0.8, а кадр із текстом —
лише ~0.1–0.3, тож текстова добавка тоне.

In [ ]:
m0 = MODELS[0]
sub = embs[m0][np.random.default_rng(0).choice(len(embs[m0]), min(2000, len(embs[m0])), replace=False)]
off = ~np.eye(len(sub), dtype=bool)
print(f"{m0}: середній cos тайл–тайл {(sub @ sub.T)[off].mean():.3f} · тайл–текст «many cars» "
      f"{(sub @ enc[m0].text_query('many cars')).mean():.3f}")

Робочий варіант — **злиття на рівні оцінок**: кожну схожість стандартизуємо по архіву (z-score) і зважуємо:
$$s_i = z\big(\cos(q_{\text{img}}, x_i)\big) + \lambda\, z\big(\cos(q_{\text{text}}, x_i)\big)$$
λ = 0 — чистий пошук за зразком, великий λ — чистий текстовий пошук.

У панелі: оберіть кадр повзунком або **«Обрати фото»** (своє JPG / PNG), протягніть мишею рамку навколо цілі, за бажанням
допишіть текст (`many cars`, `at night`, `next to a building`) і рухайте λ. Зразок вирізано з того самого архіву, тож кадри
його кліпу в межах ±5 с і вся його GT-подія виключаються з видачі — інакше це впізнавання себе, а не пошук.

In [ ]:
EXCLUDE_S = 5.0


def zscore(x):
    x = np.where(np.isfinite(x), x, np.nan)
    return np.nan_to_num((x - np.nanmean(x)) / np.nanstd(x), nan=-1e9)


def near_frames(kf_id, gt_query=None):
    r = kf.set_index("kf_id").loc[kf_id]
    excl = set(kf[(kf.source_id == r.source_id) & ((kf.t_sec - r.t_sec).abs() <= EXCLUDE_S)].kf_id.astype(int)) | {int(kf_id)}
    for ev in (gt[gt_query]["events"] if gt_query else []):
        if kf_id in ev:
            excl |= set(ev)
    return excl


def ask_image(crop, text="", lam=1.0, models=None, k=8, use_tiles=True, src_kf=None, gt_query=None):
    excl = near_frames(src_kf, gt_query) if src_kf is not None else set()
    rel = gt[gt_query]["relevant"] - excl if gt_query else None
    for name in models or MODELS:
        s = zscore(P.all_tile_scores(indices[name], enc[name].image_query(crop)))
        if text:
            s = s + lam * zscore(P.all_tile_scores(indices[name], enc[name].text_query(text)))
        res = P.top_frames(s, tiles, kf, k=k, use_tiles=use_tiles, exclude=excl)
        hit = f" | P@{k} = {np.mean([r in rel for r in res.kf_id]):.2f}" if rel is not None else ""
        P.show_results(res, f"{name}: зразок{f' + «{text}» (λ={lam:g})' if text else ''} · z-оцінка{hit}", relevant=rel)


r0 = P.top_frames(P.all_tile_scores(indices[m0], enc[m0].text_query(gt[demo_q]["text"])), tiles, kf, k=1).iloc[0]
img_panel = P.image_search_panel(ask_image, kf, MODELS, gt_texts={q: g["text"] for q, g in gt.items()},
                                 start_kf=int(r0.kf_id), start_box=(int(r0.x0), int(r0.y0), int(r0.x1), int(r0.y1)),
                                 gt_query=demo_q)

**Підсумок розділу.** Пошук за зразком знаходить «ще таких» без жодного слова, а текст-модифікатор працює лише через злиття
**оцінок**, а не векторів. λ підбирають на розміченому наборі запитів, як і пороги. Сусідні кадри одного відео заповнюють
видачу дублікатами — тому й потрібна дедуплікація та метрика по подіях.

## 6. Метрики (слайд 46 «Як міряти мультимодальну систему»)

Для кожної моделі всі запити з GT у чотирьох конфігураціях: промпт **plain** (`"a truck"`) або **aerial** (ансамбль із 6
формулювань, слайд 38), вхід **кадр** або **тайли** (слайд 23).
**EventR@50** — частка подій у перших 50 кадрах видачі (те, що реально побачить оператор); **P@10** — скільки сміття
на першому екрані; **AP** — середня точність по всьому ранжуванню.

In [ ]:
CONFIGS = {"plain": P.PLAIN_TEMPLATES, "aerial": P.AERIAL_TEMPLATES}
qvecs = {(m, mode, q): enc[m].text_query(gt[q]["text"], tpls) for m in MODELS for mode, tpls in CONFIGS.items() for q in gt}
recs = []
for (m, mode, q), qv in qvecs.items():
    ts = P.all_tile_scores(indices[m], qv)
    for use_tiles in ([False, True] if TILE_GRID else [False]):
        recs.append(dict(model=m, prompt=mode, input="tiles" if use_tiles else "frame", query=q,
                         **P.rank_metrics(P.frame_scores(ts, tiles, len(kf), use_tiles), gt[q])))
R = pd.DataFrame(recs)
summary = R.groupby(["model", "prompt", "input"])[["EventR@10", "EventR@50", "P@10", "AP"]].mean().sort_values("EventR@50", ascending=False)
best_cfg = summary.reset_index().groupby("model").first()[["prompt", "input"]]
P.plot_metrics(R, ["EventR@50", "P@10"], baselines={"P@10": np.mean([len(v["relevant"]) / len(kf) for v in gt.values()])})
display(summary)
per_q = R.merge(best_cfg.reset_index(), on=["model", "prompt", "input"]).pivot(index="query", columns="model", values="EventR@50")
per_q["prevalence"] = R.groupby("query").prevalence.first()
display(per_q.sort_values("prevalence"))

**Підсумок розділу.**
* Дивіться на EventR@50 поряд із `prevalence`, а не на картинки першої видачі.
* **Пікселі важать більше за модель:** перехід «кадр → тайли» часто дає більший приріст, ніж заміна моделі (слайд 23).
* **Домен важить, але не автоматично перемагає:** RemoteCLIP / SkyCLIP навчені переважно на надирі з великої висоти;
  на косому ракурсі низьких дронів їхня перевага не гарантована (слайд 38). Промпт із ракурсом — теж гіпотеза, яку міряють.
* Таблиця по запитах показує, де модель провалюється: зазвичай це рідкісні й дрібні цілі.

## 7. Три зони впевненості (слайди 48 «Human-in-the-loop», 49 «Три зони впевненості»)

Пороги ставимо за виміряними розподілами сирого cos на розміченому архіві, окремо для кожної моделі й запиту (найкраща
конфігурація з розділу 6). Рухайте `T_low` і `T_high`: таблиця покаже, скільки кадрів піде оператору, скільки цілей система
відкине автоматично і яка точність зони пріоритетного оповіщення. Старт: `T_low` зберігає 90 % кадрів з ціллю, `T_high`
пропускає 1 % порожніх.

In [ ]:
def get_scores(m, q):
    mode, inp = best_cfg.loc[m, "prompt"], best_cfg.loc[m, "input"]
    fs = P.frame_scores(P.all_tile_scores(indices[m], qvecs[(m, mode, q)]), tiles, len(kf), use_tiles=(inp == "tiles"))
    rel = np.zeros(len(kf), bool)
    rel[list(gt[q]["relevant"])] = True
    return fs, rel


P.zones_panel(get_scores, MODELS, {q: g["text"] for q, g in gt.items()}, sec_per_frame=2.0)

**Підсумок розділу.** Розподіли «з ціллю» і «без» перекриваються, тож будь-який поріг — компроміс між пропущеними цілями і
годинами оператора. Шкали моделей несумісні, а пороги «пливуть» з сезоном і сенсором — їх перекалібровують (слайд 49: пороги — окремо для кожного промпту й сенсора).
Зона 3 — це пріоритетний показ, а не автоматична дія: висока схожість ≠ підтверджена ціль.

## 8. Вузький канал: PQ-коди замість векторів (слайди 50 «Вузький канал», 51 «Split-computing»)

На борту лишається лише PQ-код вектора: `M × nbits / 8` байт (64 Б при `M=64, nbits=8`). Скільки пошуку втрачаємо і скільки
повертає дозапит повних векторів для топу видачі (слайд 51)? Кодову книгу вчимо на всіх тайлах архіву (у бою — на великому
архіві на землі, до вильоту); з тайлами векторів досить (FAISS радить ≥ 39 на центроїд, тобто ≥ 10 тис.); з `TILE_GRID=None` їх лише ~3 тис., і `build_pq` попередить, що втрати PQ тоді — оцінка зверху.

In [ ]:
m = summary.reset_index().iloc[0]["model"]           # найкраща модель за EventR@50
mode, inp = best_cfg.loc[m, "prompt"], best_cfg.loc[m, "input"]
use_t = inp == "tiles"
rows_sel = np.arange(len(tiles)) if use_t else np.flatnonzero(tiles.tile_idx.to_numpy() == 0)
pq, info = P.build_pq(embs[m][rows_sel], M=64, nbits=8, train=embs[m])


def to_full(s_sel):
    s = np.full(len(tiles), -np.inf, dtype=np.float32)
    s[rows_sel] = s_sel
    return s


recs = []
for q in gt:
    qv = qvecs[(m, mode, q)]
    for vname, ts in {"Flat (повні вектори)": P.all_tile_scores(indices[m], qv),
                      f"PQ {info['code_bytes']:.0f} Б": to_full(P.pq_tile_scores(pq, qv)),
                      "PQ + дозапит топ-100": to_full(P.pq_tile_scores(pq, qv, embs[m][rows_sel], rerank_top=100 * (tpf if use_t else 1)))}.items():
        recs.append(dict(variant=vname, query=q, **P.rank_metrics(P.frame_scores(ts, tiles, len(kf), use_t), gt[q])))
print(f"{m} ({mode}/{inp}) · {info}")
display(pd.DataFrame(recs).groupby("variant", sort=False)[["EventR@50", "P@10", "AP"]].mean())

meta_b = 16
msg, fp16 = info["code_bytes"] + meta_b, info["fp16_bytes"] + meta_b
print(f"Повідомлення: PQ {msg:.0f} Б (код {info['code_bytes']:.0f} + метадані {meta_b}) проти FP16 {fp16:.0f} Б")
pd.DataFrame([{"канал, кбіт/с": kbps, "PQ-повідомлень/с": kbps * 125 / msg, "FP16-векторів/с": kbps * 125 / fp16}
              for kbps in (1.5, 3.0, 6.0)])

**Підсумок розділу.** PQ + дозапит топу повертає більшу частину якості при десятках байт на кадр: у канал 1.5–6 кбіт/с
вміщаються кілька PQ-повідомлень на секунду, а FP16-векторів — менше одного. Головна вимога — **один спільний простір**:
image tower на борту і text encoder на землі з однієї моделі, кодова книга спільна (слайд 51).

## Підсумок

1. **Пошук без розмітки працює:** текстовий запит або зразок звужує години відео до кількох десятків кадрів (слайди 41–42, 48).
2. **Пікселі важливіші за модель:** тайли і letterbox замість стискання кадру; домен і промпт — гіпотези, які міряють (слайди 22–23, 38).
3. **Чесні метрики:** EventR@k і P@k поряд із `prevalence`, окремо для кожного запиту (слайд 46).
4. **Пороги — по сирому cos**, окремо для моделі й запиту; три зони розподіляють роботу між автоматикою і людиною (слайд 49).
5. **Вузький канал:** по радіо йдуть 80-байтні PQ-повідомлення, пошук і рішення — на землі (слайди 50–51).

**Самостійно:** додайте 3 власні запити до GT; порівняйте `SIZE="large"`; поставте `TILE_GRID=(3, 4)` і порахуйте, скільки
пікселів займає ціль після ресайзу тайла; повторіть розділ 8 з `M=32`.